# Ficha Técnica y Control de Calidad: Windsor Housing Dataset

Este notebook implementa la validación técnica, extracción de metadatos, diccionario de datos y auditoría de calidad sobre el dataset ubicado en `data/raw/housing.csv`.

## 0. Importación de Librerías y Carga de Datos

In [ ]:
import hashlib
from pathlib import Path
import pandas as pd
import numpy as np

# Ruta al dataset en la capa raw / bronce
DATA_PATH = Path("data/raw/housing.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo en: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
# Si el CSV contiene una columna índice sin nombre (proveniente de R / Rdatasets)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print(f"Dataset cargado exitosamente. Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas.")
df.head()

## 1. Procedencia e Integridad (Checksum SHA-256)

In [ ]:
def calculate_sha256(filepath: Path) -> str:
    sha256_hash = hashlib.sha256()
    with open(filepath, "rb") as f:
        for byte_block in iter(lambda: f.read(65536), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

provenance_metadata = {
    "Dataset": "House Prices in the City of Windsor, Canada (1987)",
    "Fuente Original": "Journal of Applied Econometrics Data Archive",
    "URL Fuente": "http://qed.econ.queensu.ca/jae/1996-v11.6/anglin-gencay/",
    "Referencia": "Anglin, P., & Gencay, R. (1996). Semiparametric Estimation of a Hedonic Price Function. JAE, 11, 633-648.",
    "Archivo Local": str(DATA_PATH),
    "SHA-256 Checksum": calculate_sha256(DATA_PATH)
}

df_provenance = pd.DataFrame(list(provenance_metadata.items()), columns=["Atributo", "Valor"])
df_provenance

## 2. Licencia y Marco Legal

In [ ]:
license_info = {
    "Tipo de Licencia": "GNU General Public License (GPL-2 / GPL-3) via R AER package",
    "Clasificación": "Open Data / Investigación Académica",
    "Uso Permitido": "Libre uso educativo, investigación estadística y modelado econométrico.",
    "Requisitos de Atribución": "Citar formalmente a Anglin & Gencay (1996).",
    "Restricciones de Privacidad": "Datos agregados y anonimizados a nivel de inmueble sin PII (Personally Identifiable Information)."
}

df_license = pd.DataFrame(list(license_info.items()), columns=["Término Legal", "Detalle"])
df_license

## 3. Diccionario de Datos Automatizado

In [ ]:
# Descripción semántica de las variables según el paper/documentación
field_descriptions = {
    'price': 'Precio final de venta del inmueble (CAD)',
    'lotsize': 'Tamaño del lote de la propiedad en pies cuadrados (sq ft)',
    'bedrooms': 'Número de dormitorios',
    'bathrooms': 'Número de cuartos de baño completos',
    'stories': 'Número de pisos de la casa (excluye sótano)',
    'driveway': '¿Posee entrada vehicular pavimentada? (yes/no)',
    'recreation': '¿Posee salón de juegos / recreación? (yes/no)',
    'fullbase': '¿Tiene sótano completamente acondicionado/terminado? (yes/no)',
    'gasheat': '¿Usa gas para el calentamiento de agua? (yes/no)',
    'aircon': '¿Cuenta con sistema de aire acondicionado central? (yes/no)',
    'garage': 'Capacidad de plazas de estacionamiento en el garaje',
    'prefer': '¿Está localizada en el vecindario de alta preferencia de la ciudad? (yes/no)'
}

# Construcción programática del diccionario de datos
data_dictionary = []
for col in df.columns:
    col_dtype = str(df[col].dtype)
    n_unique = df[col].nunique()
    sample_vals = str(df[col].dropna().unique()[:4].tolist())
    desc = field_descriptions.get(col, 'Sin descripción disponible')
    
    # Clasificación estadística
    if pd.api.types.is_numeric_dtype(df[col]):
        stat_type = "Numérica Discreta" if n_unique < 15 else "Numérica Continua"
    else:
        stat_type = "Categórica (Factor/Binaria)"
        
    data_dictionary.append({
        'Variable': col,
        'Tipo Pandas': col_dtype,
        'Tipo Estadístico': stat_type,
        'Valores Únicos': n_unique,
        'Ejemplos': sample_vals,
        'Descripción': desc
    })

df_dict = pd.DataFrame(data_dictionary)
df_dict

## 4. Auditoría de Calidad del Dato (Data Quality Profiling)

In [ ]:
# 4.1 Completitud y Registros Duplicados
null_counts = df.isnull().sum()
null_percent = (null_counts / len(df)) * 100
duplicates = df.duplicated().sum()

quality_summary = pd.DataFrame({
    'Columna': df.columns,
    'Nulos': null_counts.values,
    '% Nulos': null_percent.values,
    'Completitud %': 100 - null_percent.values
})

print(f"=== RESUMEN GLOBAL DE CALIDAD ===")
print(f"Total Filas: {len(df)}")
print(f"Filas Duplicadas: {duplicates}")
print(f"Total Valores Nulos en el dataset: {null_counts.sum()}")
quality_summary

In [ ]:
# 4.2 Consistencia de Variables Categóricas
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

cat_audit = []
for c in cat_cols:
    unique_vals = set(df[c].dropna().unique())
    is_valid_binary = unique_vals.issubset({'yes', 'no'})
    cat_audit.append({
        'Variable': c,
        'Valores Observados': list(unique_vals),
        'Consistente (yes/no)': is_valid_binary
    })

df_cat_audit = pd.DataFrame(cat_audit)
print("=== AUDITORÍA DE CONSISTENCIA CATEGÓRICA ===")
df_cat_audit

In [ ]:
# 4.3 Verificación de Rangos e Integridad Numérica
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_summary = df[num_cols].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]

# Comprobación de anomalías físicas (ej. valores negativos imposibles)
num_summary['Valores_Negativos'] = (df[num_cols] < 0).sum()

print("=== ESTADÍSTICAS Y CONTROL DE RANGO NUMÉRICO ===")
num_summary